<a href="https://colab.research.google.com/github/heldigard/Electiva-II-Analitica-IA/blob/master/clases/ClaseVirtual-API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌐 Clase Virtual: APIs para Análisis de Datos

📅 **Clase pregrabada — mírala ANTES del lunes 7 de septiembre de 2026** (entre la Clase 2 del 24 de agosto y la Clase 3)
⏱️ **Duración aproximada:** 45 minutos, a tu propio ritmo
💡 **Tip:** si tienes dudas de este material, la **holgura del 31 de agosto** es el espacio perfecto para resolverlas

## Universidad UdeColombia - Especialización en Analítica de Datos

---

### 🎯 Objetivo de la clase

Al finalizar esta sesión, serás capaz de:
- ✅ Entender qué es una API y por qué es una herramienta poderosa para el análisis de datos
- ✅ Consultar APIs gratuitas utilizando Python desde Google Colab
- ✅ Procesar los datos obtenidos (en formato JSON) y convertirlos en un DataFrame
- ✅ Realizar un análisis exploratorio básico sobre la información recolectada

> **Nota:** Esta clase está disponible en video pregrabado. El notebook sirve como material de apoyo y referencia.

---

#### **1. Introducción a las APIs: Las Puertas de Acceso a los Datos del Mundo**

API (Interfaz de Programación de Aplicaciones). Una buena analogía es pensar en una API como un mesero en un restaurante:

*   **Usted (el cliente/analista):** Quiere obtener algo específico (un plato de comida/datos).
*   **La Cocina (el servidor/fuente de datos):** Tiene lo que usted necesita, pero no puede entrar directamente a la cocina a prepararlo.
*   **El Mesero (la API):** Toma su pedido (petición), lo lleva a la cocina y le trae de vuelta lo que solicitó (respuesta).

En el mundo digital, las APIs permiten que diferentes aplicaciones se comuniquen entre sí e intercambien datos de una manera estandarizada. Para un analista de datos, esto significa acceso a datos en tiempo real de diversas fuentes: redes sociales, mercados financieros, información gubernamental, datos meteorológicos y mucho más.

#### **2. Selección de APIs Gratuitas para Proyectos de Análisis**

No todas las APIs son iguales. Para empezar, es ideal utilizar APIs que no requieran autenticación compleja (como OAuth) y que ofrezcan datos interesantes para analizar. Aquí tiene una selección de APIs excelentes para principiantes:

| API | Descripción de los Datos | Requiere Registro | Ideal Para... |
| --- | --- | --- | --- |
| **JSONPlaceholder** | Proporciona datos falsos pero realistas (usuarios, posts, fotos, etc.) en formato JSON. | No | Aprender a hacer peticiones y procesar la estructura de los datos. |
| **PokéAPI** | Una base de datos completa sobre el universo Pokémon: criaturas, habilidades, movimientos, etc. | No | Practicar con datos bien estructurados y relaciones entre diferentes "endpoints". |
| **OpenWeatherMap** | Ofrece datos meteorológicos actuales y pronósticos para ciudades de todo el mundo. | Sí (para obtener una API Key gratuita) | Trabajar con datos numéricos, series de tiempo y geolocalización. |
| **CountriesNow** | Datos de países, incluidos históricos de población, en JSON anidado. | No | Practicar JSON anidado, limpieza y análisis demográfico. |
| **TheCocktailDB** | Una base de datos de recetas de cócteles con ingredientes, imágenes e instrucciones. | No para pruebas (usa la clave de desarrollo `1`); sí para uso serio | Análisis de texto y datos categóricos. |

Para esta clase, usarás **CountriesNow**: no requiere clave y su respuesta incluye una lista anidada de históricos de población.

#### **3. ¡Manos a la Obra! Consultando una API con Python en Google Colab**

Ahora sigue estos pasos en un nuevo notebook de Google Colab.



**Paso 1: Usar la librería `requests`**

En Google Colab, `requests` y `pandas` ya están disponibles; no necesitas instalar nada para esta actividad.



In [ ]:
# En Google Colab, requests ya viene instalada. Continúa con las importaciones.

**Paso 2: Importar las librerías necesarias**

Necesitaremos `requests` para hacer la petición a la API y `pandas` para manejar los datos.

In [ ]:
import requests
import pandas as pd

**Paso 3: Realizar la petición a la API**

Vamos a pedir a CountriesNow los históricos de población de todos los países. Cada país contiene una lista anidada llamada `populationCounts`.

In [ ]:
# Endpoint público con históricos de población (países + agregados del Banco Mundial)
url = "https://countriesnow.space/api/v0.1/countries/population"

# El timeout evita que el notebook espere indefinidamente si la red falla.
response = requests.get(url, timeout=20)

# Si la respuesta HTTP no es exitosa, detenemos el flujo con un error claro.
response.raise_for_status()
payload = response.json()

# Validamos la forma esperada antes de usar los datos.
if not isinstance(payload, dict) or payload.get('error') or not isinstance(payload.get('data'), list):
    raise ValueError(f"Respuesta inesperada de la API: {payload}")

print(f"✅ Respuesta válida: {len(payload['data'])} registros recibidos (incluye países y algunos agregados regionales).")


**Paso 4: Convertir la respuesta a formato JSON**

La mayoría de las APIs modernas devuelven datos en formato JSON (JavaScript Object Notation). En esta respuesta, `data` es una lista de países y cada país guarda otra lista con sus registros de población.

In [ ]:
# Extraemos la lista validada en el paso anterior.
data = payload['data']

# El primer registro suele ser un agregado regional (p. ej. "Arab World").
# Para ver la forma anidada, buscamos un país real de ejemplo:
ejemplo = next((row for row in data if row.get('iso3') == 'COL'), data[0])
print("Ejemplo de registro (Colombia si está disponible):")
print(ejemplo)


**Paso 5: Cargar los datos en un DataFrame de Pandas**

¡Aquí es donde comienza la magia del análisis de datos! Un DataFrame nos proporciona una estructura tabular con la que es mucho más sencillo trabajar.

In [ ]:
# Creamos el DataFrame directamente desde la lista de diccionarios
df_paises = pd.DataFrame(data)

# Primeras filas: verás agregados regionales mezclados con países.
# Eso es normal en esta fuente; más adelante los filtraremos.
df_paises.head()


#### **4. Exploración y Análisis de Datos (EDA)**

Una vez que los datos están en un DataFrame, podemos empezar a hacer preguntas.

**Pregunta 1: ¿Cuáles son las columnas que tenemos y qué tipo de datos contienen?**



In [ ]:
# Obtenemos información general del DataFrame
df_paises.info()

*Observación:* La columna `populationCounts` contiene listas de diccionarios. Antes de ordenar países por población, la convertiremos en filas para quedarnos con el registro más reciente de cada uno.

> ⚠️ **Calidad de datos:** esta API mezcla **países** con **agregados** (por ejemplo *World*, *Middle income*, *East Asia & Pacific*). Antes de un ranking de "países más poblados", hay que **filtrar** esos agregados; si no, el top 10 será engañoso. Es un ejemplo real de por qué validar siempre la fuente.


**Pregunta 2: ¿Cuáles son los 10 países más poblados según esta fuente?**

> ⚠️ **Vigencia de la fuente — lee esto antes de interpretar el resultado.** El histórico que devuelve esta API
> termina alrededor de **2018** (y no todos los países comparten el mismo último año disponible). Por eso la
> pregunta correcta no es "¿cuáles son los más poblados **del mundo**?" sino "¿cuáles son los más poblados
> **según esta fuente y en el último año que ella reporta**?".
>
> Ejemplo de por qué importa: con estos datos China aparece por encima de India, pero las estimaciones de la
> ONU sitúan a **India como el país más poblado desde 2023**. El dato no está "mal" — está **desactualizado**,
> y confundir ambas cosas es uno de los errores más costosos del análisis. **Siempre revisa la fecha de corte
> de tu fuente antes de escribir una conclusión en presente.**

In [ ]:
# Convertimos la lista anidada en filas y expandimos cada diccionario.
df_historico = df_paises.explode('populationCounts').dropna(subset=['populationCounts'])
df_historico = pd.concat(
    [df_historico[['country', 'iso3']].reset_index(drop=True),
     pd.json_normalize(df_historico['populationCounts']).reset_index(drop=True)],
    axis=1
)
df_historico['value'] = pd.to_numeric(df_historico['value'], errors='coerce')
df_historico['year'] = pd.to_numeric(df_historico['year'], errors='coerce')
df_historico = df_historico.dropna(subset=['value', 'year'])

# Filtrar agregados del Banco Mundial (regiones / grupos de ingreso).
# Sin este paso, el "top 10 países" incluiría filas como World o Middle income.
agregados = (
    df_historico['country'].str.contains(
        r"World|income|dividend|IDA|IBRD|OECD|HIPC|excluding|classified|"
        r"small states|Euro area|European Union|Fragile|blend|demographic|"
        r"East Asia|Europe &|Latin America|Middle East|North America|"
        r"South Asia|Sub-Saharan|Pacific island|Arab World|"
        r"Central Europe and the Baltics|Not classified|developed countries",
        case=False,
        na=False,
        regex=True,
    )
)
df_paises_reales = df_historico.loc[~agregados].copy()
print(f"Registros históricos: {len(df_historico):,} → países (sin agregados): {len(df_paises_reales):,}")

# Conservamos el dato más reciente disponible para cada país.
df_actual = (
    df_paises_reales.sort_values('year')
    .drop_duplicates(subset='country', keep='last')
)
top_10_poblacion = df_actual.nlargest(10, 'value')
print(top_10_poblacion[['country', 'year', 'value']].rename(columns={'value': 'population'}))

# Verifica la vigencia: ¿de qué años son realmente estas cifras?
anios_top = sorted(int(a) for a in top_10_poblacion['year'].unique())
print(f"\n⚠️ Años de corte presentes en este top 10: {anios_top}")
print(f"   Último año disponible en toda la fuente: {int(df_paises_reales['year'].max())}")
print("   Si los años no coinciden entre países, el ranking compara cifras de momentos distintos.")


*Observación:* El JSON ya se transformó en una tabla: cada fila representa un país y un año. Ahora dejaremos nombres de columnas más claros para presentar el resultado. Fíjate en que la columna `anio` viaja junto al dato: **un número de población sin su año no es comunicable.**

**Preparando una tabla para comunicar el resultado:**

In [ ]:
top_10_presentacion = top_10_poblacion.rename(columns={
    'country': 'pais', 'year': 'anio', 'value': 'poblacion'
})
print(top_10_presentacion[['pais', 'anio', 'poblacion']])

**Pregunta 3: ¿Cuál es la población acumulada de esos 10 países?**

> 🔎 Antes de sumar, comprueba en la salida anterior que los 10 registros correspondan **al mismo año**. Sumar
> cifras de años distintos produce un total que no describe ningún momento real.

In [ ]:
poblacion_top_10 = top_10_poblacion['value'].sum()
print(f"Población acumulada del top 10: {poblacion_top_10:,.0f} personas")

**Visualización de los resultados**

Una parte crucial del análisis es la visualización. Usemos `matplotlib` para comparar la población de los 10 países con mayor población disponible.


In [ ]:
import matplotlib.pyplot as plt

# Creamos un gráfico de barras con los resultados del top 10.
top_10_grafico = top_10_poblacion.sort_values('value')
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_10_grafico['country'], top_10_grafico['value'])

anio_corte = int(top_10_grafico['year'].max())
ax.set_title(f'Top 10 países por población — datos de {anio_corte} (sin agregados regionales)')
ax.set_xlabel('Población')
ax.set_ylabel('País')

# Miles de millones son ilegibles en notación científica: formateamos el eje
ax.xaxis.set_major_formatter(lambda x, pos: f'{x/1e6:,.0f} M')

plt.tight_layout()
plt.show()

#### **5. Propuestas de Proyectos**

Para afianzar lo aprendido, elige uno de los siguientes proyectos para trabajarlo de forma individual o en grupo:

1.  **Analista de Pokémon (PokéAPI):**
    *   Obtener los datos de los primeros 151 Pokémon.
    *   Analizar la distribución de tipos (Fuego, Agua, Planta, etc.).
    *   ¿Existe alguna correlación entre el peso, la altura y las estadísticas base (ataque, defensa)?
    *   Visualizar las estadísticas promedio por cada tipo principal.

2.  **Meteorólogo de Ciudades Globales (OpenWeatherMap API):**
    *   Elegir una lista de 15-20 ciudades de diferentes continentes.
    *   Consultar la API para obtener el clima actual (temperatura, humedad, velocidad del viento).
    *   ¿Qué ciudad es la más cálida y la más fría en este momento?
    *   Crear un gráfico de barras comparando las temperaturas.

3.  **Explorador de Datos Falsos (JSONPlaceholder):**
    *   Obtener la lista de usuarios y sus posts.
    *   ¿Qué usuario ha escrito más posts?
    *   Calcular el número promedio de posts por usuario.
    *   Visualizar la distribución geográfica de los usuarios (usando la latitud y longitud que proveen).

#### ✅ Checkpoint final

Antes de terminar, verifica que puedes explicar: (1) qué validó `raise_for_status()`, (2) por qué comprobamos la forma del JSON y (3) cómo `explode()` convirtió una lista anidada en filas.

#### Plan de contingencia

Si la consulta falla, ejecuta primero la celda de petición otra vez y revisa el mensaje de error. Si continúa fallando por red o por una respuesta inesperada, guarda el error, avisa al profesor y usa el ejemplo impreso en la celda de JSON para continuar identificando la estructura; no continúes con datos no validados.
